# STAGE 12 · CheXbert evaluation — making the clinical score comparable

### ~25 min · ~0.75 CU · every checkpoint read-only

---

## Why this exists

Every clinical-efficacy number reported so far (0.5799 Stage 4, 0.5937 Stage 11) came from a **regex extractor written for this project**. It agrees with our manifest labels at F1 0.9203, so it is not unreasonable — but **nobody else uses it**, so those numbers cannot be placed beside any published result.

**CheXbert** ([Smit et al., EMNLP 2020](https://aclanthology.org/2020.emnlp-main.117.pdf)) is the BERT-based labeller the field actually uses. Running it converts our number from *internally consistent* to *externally comparable*.

## What it also fixes

The full generated reports were never saved — Stage 4B and Stage 11 each wrote only 100 samples. This notebook regenerates all 4,722 for **both** models and saves them permanently, so no future evaluation needs a GPU again.

| output | |
|---|---|
| `reports_stage4_test.txt` | 4,722 generated reports |
| `reports_stage11_test.txt` | 4,722 generated reports |
| `references_test.txt` | 4,722 ground-truth reports |
| `stage12_chexbert.json` | all scores |

## 🔒 Nothing is trained

Both checkpoints are opened read-only and SHA-256 verified before and after.

---
# 0 · Setup

In [ ]:
import os, sys, json, time, hashlib, warnings, subprocess
from pathlib import Path
warnings.filterwarnings('ignore')

# f1chexbert bundles the CheXbert labeller and pulls weights from HuggingFace
# (StanfordAIMI/RRG_scorers) -- public, no Stanford access form required.
try:
    import f1chexbert  # noqa: F401
except ImportError:
    print('  installing f1chexbert ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'f1chexbert'], check=True)
try:
    from rouge_score import rouge_scorer  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rouge-score'], check=True)
print('  packages ready')

import numpy as np, pandas as pd, torch
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
else: print('  Drive already mounted')
PROJECT  = Path('/content/drive/MyDrive/Component_01')
IMG_ROOT = Path('/content/cardio_image_384')
TAR      = PROJECT / 'data' / 'images' / 'cardio_384.tar'
MANIFEST = PROJECT / 'training_manifest'
S4_CKPT  = PROJECT / 'checkpoints' / 'stage4'  / 'best.pt'
S5_CKPT  = PROJECT / 'checkpoints' / 'stage5'  / 'best.pt'
S11_CKPT = PROJECT / 'checkpoints' / 'stage11' / 'best.pt'
S6CACHE  = PROJECT / 'reports' / 'stage6' / 'cache'
OUT      = PROJECT / 'reports' / 'stage12'; OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT))

SHA = {n: hashlib.sha256(p.read_bytes()).hexdigest()
       for n, p in (('stage4', S4_CKPT), ('stage5', S5_CKPT), ('stage11', S11_CKPT))}
for n, h in SHA.items(): print('  %-8s sha %s' % (n, h[:40]))
DEV = 'cuda'; assert torch.cuda.is_available(), 'select an L4 GPU'
print(' ', subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

---
# 0b · Stage the images

In [ ]:
if not IMG_ROOT.exists():
    import shutil
    t0 = time.time(); lt = Path('/content/cardio_384.tar')
    if not lt.exists():
        assert TAR.exists(), 'tar not found: ' + str(TAR)
        shutil.copy(TAR, lt)
    subprocess.run(['tar', '-xf', str(lt), '-C', '/content'], check=True)
    print('  staged in %.1f min' % ((time.time() - t0) / 60))
    try: lt.unlink()
    except OSError: pass
if not IMG_ROOT.exists():
    cand = [p for p in Path('/content').glob('*') if p.is_dir() and (p/'test').is_dir()]
    assert cand, 'no extracted directory containing test/'
    IMG_ROOT = cand[0]
print('  images:', IMG_ROOT.exists())

---
# 1 · Generate all 4,722 reports from both models

Greedy decoding, matching Stage 4B's ablation winner. Cached — re-running skips this.

In [ ]:
import stage6_acr as acr, stage9_fairness as s9, stage9b_gradrev as s9b
import stage11_conditioned as s11
from cxr_transforms import build_transform
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from PIL import Image
PATH = s11.PATHOLOGIES
TOK  = AutoTokenizer.from_pretrained('GanjinZero/biobart-v2-base')
TF   = build_transform('test')
te   = pd.read_csv(MANIFEST / 'manifest_test.csv', low_memory=False)
REFS = [' '.join(str(t).split()) for t in te.report]
print('  test reports:', len(REFS))

# classifier probabilities -> prompts (Stage 11 only)
PRt = np.load(S6CACHE / 'probs_test.npy')
PRv = np.load(S6CACHE / 'probs_val.npy')
va  = pd.read_csv(MANIFEST / 'manifest_val.csv', low_memory=False)
LV  = va[PATH].astype(int)
THR = [s9.best_f1_threshold(LV[k].values, PRv[:, j]) for j, k in enumerate(PATH)]

class TestDS(Dataset):
    def __init__(self, df, probs, thr, prompt):
        self.df, self.probs, self.thr, self.prompt = df.reset_index(drop=True), probs, thr, prompt
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        img = TF(Image.open(IMG_ROOT / self.df.image_path[i]))
        pr  = s11.build_prompt(self.probs[i], self.thr) if self.prompt else ''
        return img, pr

def collate(b):
    imgs, prompts = zip(*b)
    pid, pmask = s11.encode_prompts(prompts, TOK) if any(prompts) else (None, None)
    return torch.stack(imgs), pid, pmask

@torch.no_grad()
def gen_reports(ckpt, use_prompt, tag):
    f = OUT / ('reports_' + tag + '_test.txt')
    if f.exists():
        R = f.read_text(encoding='utf-8').split('\n')[:len(REFS)]
        if len(R) == len(REFS):
            print('  %-8s cached (%d)' % (tag, len(R))); return R
    m = s11.CXRConditionedGenerator('GanjinZero/biobart-v2-base', 0.1, 1)
    ck = torch.load(ckpt, map_location='cpu', weights_only=False)
    if 'ema' in ck: m.load_stage4(ck, use_ema=True)
    else:           m.load_state_dict(ck['model'])
    del ck; m = m.eval().to(DEV)
    dl = DataLoader(TestDS(te, PRt, THR, use_prompt), batch_size=16, shuffle=False,
                    num_workers=2, collate_fn=collate)
    out, t0 = [], time.time()
    for x, pid, pmask in dl:
        x = x.to(DEV, non_blocking=True)
        a = pid.to(DEV) if pid is not None else None
        b_ = pmask.to(DEV) if pmask is not None else None
        with torch.autocast('cuda', dtype=torch.bfloat16):
            ids = m.generate(x, a, b_, num_beams=1, max_length=192,
                             min_length=24, no_repeat_ngram_size=3)
        out += [' '.join(t.split()) for t in TOK.batch_decode(ids, skip_special_tokens=True)]
        print('\r  %-8s %d/%d' % (tag, len(out), len(REFS)), end='')
    f.write_text('\n'.join(out), encoding='utf-8')
    print('\r  %-8s %d reports in %.1f min' % (tag, len(out), (time.time()-t0)/60))
    del m; torch.cuda.empty_cache()
    return out

(OUT / 'references_test.txt').write_text('\n'.join(REFS), encoding='utf-8')
GEN = {'stage4':  gen_reports(S4_CKPT,  False, 'stage4'),
       'stage11': gen_reports(S11_CKPT, True,  'stage11')}
for k, v in GEN.items(): assert len(v) == len(REFS), k + ' length mismatch'
print('\n  saved to', OUT)

---
# 2 · CheXbert

BERT-base labeller, 14 CheXpert findings. Weights come from HuggingFace (`StanfordAIMI/RRG_scorers`) — public, no access form.

> The shim below restores `encode_plus`, which transformers ≥5 removed but `f1chexbert` still calls. Without it the labeller raises `AttributeError`.

In [ ]:
from transformers import BertTokenizer
if not hasattr(BertTokenizer, 'encode_plus'):
    def _ep(self, tokens, **kw):
        ids = (self.convert_tokens_to_ids(tokens) if isinstance(tokens, (list, tuple))
               else self.encode(tokens, add_special_tokens=False))
        return {'input_ids': [self.cls_token_id] + list(ids) + [self.sep_token_id]}
    BertTokenizer.encode_plus = _ep
    print('  transformers>=5 shim installed')

# f1chexbert downloads chexbert.pth into a NESTED HuggingFace cache but then
# looks for it at a FLAT path, so construction fails with FileNotFoundError even
# though the 1.3 GB file is on disk. Link it into place first.
import glob, shutil
from f1chexbert.f1chexbert import CACHE_DIR
os.makedirs(CACHE_DIR, exist_ok=True)
_dst = os.path.join(CACHE_DIR, 'chexbert.pth')
if not os.path.exists(_dst):
    _hits = [h for h in glob.glob(os.path.join(CACHE_DIR, '**', 'chexbert.pth'),
                                  recursive=True) if h != _dst]
    if not _hits:
        from huggingface_hub import hf_hub_download
        _hits = [hf_hub_download('StanfordAIMI/RRG_scorers', 'chexbert.pth',
                                 cache_dir=CACHE_DIR)]
    try:
        os.symlink(_hits[0], _dst)          # avoids a second 1.3 GB copy
    except OSError:
        shutil.copy(_hits[0], _dst)
print('  weights in place: %.0f MB' % (os.path.getsize(_dst) / 1e6))

from f1chexbert import F1CheXbert
t0 = time.time()
CB = F1CheXbert(device=DEV)
print('  CheXbert loaded in %.1f min' % ((time.time()-t0)/60))

RES = {}
for tag, hyps in GEN.items():
    t0 = time.time()
    acc, acc5, cr, cr5 = CB(hyps=hyps, refs=REFS)
    RES[tag] = dict(
        micro_f1_14=float(cr['micro avg']['f1-score']),
        macro_f1_14=float(cr['macro avg']['f1-score']),
        micro_f1_5=float(cr5['micro avg']['f1-score']),
        macro_f1_5=float(cr5['macro avg']['f1-score']),
        precision_14=float(cr['micro avg']['precision']),
        recall_14=float(cr['micro avg']['recall']),
        per_class={k: {kk: float(vv) for kk, vv in v.items()}
                   for k, v in cr.items() if isinstance(v, dict)})
    print('  %-8s CheXbert micro-F1(14) %.4f  micro-F1(5) %.4f   [%.1f min]'
          % (tag, RES[tag]['micro_f1_14'], RES[tag]['micro_f1_5'], (time.time()-t0)/60))

---
# 3 · Results ★

In [ ]:
print('=' * 88)
print('  CheXbert CLINICAL EFFICACY  (test set n=%d)' % len(REFS))
print('=' * 88)
print('  %-12s %14s %14s %12s %12s' % ('model', 'micro-F1(14)', 'micro-F1(5)',
      'precision', 'recall'))
print('  ' + '-' * 85)
for tag in ('stage4', 'stage11'):
    r = RES[tag]
    print('  %-12s %14.4f %14.4f %12.4f %12.4f'
          % (tag, r['micro_f1_14'], r['micro_f1_5'], r['precision_14'], r['recall_14']))
print('  ' + '-' * 85)
d = RES['stage11']['micro_f1_14'] - RES['stage4']['micro_f1_14']
print('  Stage 11 - Stage 4 (14-class micro-F1): %+.4f' % d)
print()
print('  Our internal regex extractor said:  0.5799 -> 0.5937  (+0.0138)')
print('  CheXbert says:                      %.4f -> %.4f  (%+.4f)'
      % (RES['stage4']['micro_f1_14'], RES['stage11']['micro_f1_14'], d))
print()
if d > 0.005:
    print('  >>> CheXbert AGREES the Stage 11 checkpoint is better.')
elif d < -0.005:
    print('  >>> CheXbert DISAGREES. Ship Stage 4 and report this honestly.')
else:
    print('  >>> CheXbert sees no meaningful difference. Either checkpoint is defensible;')
    print('      say so rather than claiming an improvement the standard metric denies.')

print('\n  per-finding F1 (Stage 11, 14-class)')
print('  %-30s %8s %8s %8s' % ('finding', 'F1', 'prec', 'recall'))
for k, v in RES['stage11']['per_class'].items():
    if k in ('micro avg', 'macro avg', 'weighted avg', 'samples avg'): continue
    print('  %-30s %8.4f %8.4f %8.4f' % (k, v['f1-score'], v['precision'], v['recall']))

---
# 4 · Save & verify

In [ ]:
from datetime import datetime
res = dict(stage=12, timestamp=datetime.now().isoformat(), n_test=len(REFS),
           chexbert=RES, internal_regex=dict(stage4=0.5799, stage11=0.5937),
           note='CheXbert micro-F1 over 14 CheXpert findings; comparable to published work')
(OUT / 'stage12_chexbert.json').write_text(json.dumps(res, indent=2, default=float),
                                           encoding='utf-8')
print('  saved', OUT / 'stage12_chexbert.json')
print('  reports saved for reuse -- no GPU needed for future evaluation')
for n, p in (('stage4', S4_CKPT), ('stage5', S5_CKPT), ('stage11', S11_CKPT)):
    assert hashlib.sha256(p.read_bytes()).hexdigest() == SHA[n], n + ' MODIFIED!'
    print('  *** %-8s VERIFIED BYTE-IDENTICAL ***' % n)

---
# What this gives you

A clinical-efficacy number computed with **the labeller the field uses**, so it can sit in a table beside published results. Report the **14-class micro-F1** — that is the standard.

⚠️ Still not a leaderboard comparison: our split is patient-disjoint but not the official MIMIC-CXR test split, and our reference text is Stage-1 cleaned. CheXbert removes the *labeller* mismatch, not the *data* mismatch.